# Patent analysis

In [135]:
import pandas as pd

In [136]:
# AltairSaver = altair_save_utils.AltairSaver()

In [137]:
from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import plotting_utils as pu

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import utils
import importlib
importlib.reload(utils);

## Load data

In [138]:
# Labelled data
data_df = utils.load_patents_data().query("topics != 'arts'")

In [139]:
# Taxonomy dataframe
topics_df = utils.load_topic_data()

In [140]:
# Transform to one id and topic pair per row
data_exploded_df = utils.explode_data(data_df).query("topic != 'arts'")

## Baseline trends

Baseline trends for patent counts

In [141]:
importlib.reload(utils);
baseline_df = utils.get_baseline_patents()

In [142]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = baseline_df,
    year_start = 2019,
    year_end = 2023  
)
trends_baseline

,magnitude,growth
counts,7572871.2,31.192759


In [143]:
fig = pu.ts_smooth(
    baseline_df.assign(Total="Total").assign(counts = lambda df: df.counts/1e+6),
    ["Total"],
    variable= "counts",
    variable_title = "Publications (millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

## Insight 0: Overall trends

Early-years project growth of funding and project counts trends



In [144]:
ts_counts = utils.get_timeseries(data_df, column='id')

In [145]:
ts_counts

,year,counts
0,2013,493
1,2014,563
2,2015,945
3,2016,1023
4,2017,1105
5,2018,1293
6,2019,1386
7,2020,1498
8,2021,1571
9,2022,1382


In [146]:
au.ts_magnitude_growth_(
    ts_df = ts_counts,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
counts,1386.2,6.950317


In [147]:
fig = pu.ts_smooth(
    ts_counts.assign(Total="Total"),
    ["Total"],
    variable= "counts",
    variable_title = "Publications",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

In [148]:
utils.get_data_distribution(data_exploded_df.query("year >= 2019"), column='type', values=['id'])

,type,counts,counts_prop
0,Biosciences,233,0.034
1,Child care & preschool,374,0.054
2,Development & learning,466,0.067
3,General,5282,0.762
4,Health,2115,0.305
5,Parenting,15,0.002
6,Social,30,0.004
7,Technology,1949,0.281


In [149]:
importlib.reload(utils);
ts_df = (
    utils.get_data_distribution(data_exploded_df, column='type', values=['id'], ts=True)
    .query("type != 'General'")
)
utils.get_data_magnitude_growth(data_exploded_df, ids=None, column='type', value='id')

,magnitude,growth,type,counts
5,6.0,162.500000,Social,30
7,3.0,125.000000,Parenting,15
2,93.2,39.000000,Development & learning,466
1,74.8,38.323353,Child care & preschool,374
0,46.6,17.322835,Biosciences,233
3,1056.4,5.805114,General,5282
6,389.8,0.974314,Technology,1949
4,423.0,-1.378751,Health,2115


In [150]:
fig = pu.ts_smooth(
    ts_df,
    ts_df['type'].unique(),
    variable= "counts",
    variable_title = "",
    category_column = 'type',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 1: Technology trends

- Magnitude and growth for technology topic overall
- Distribution of different technologies
- Growth of different technologies in UKRI funding


### Overall technology topic growth

In [151]:
tech_subtypes = set(topics_df.query("type == 'Technology'").subtype.unique())
tech_subtypes

{'AI', 'Immersive tech', 'Internet', 'Mobile'}

In [152]:
tech_type_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'type'])
)

In [153]:
# ts_amounts_tech = utils.get_timeseries(tech_type_df, column='amount')
ts_counts_tech = utils.get_timeseries(tech_type_df, column='id')
utils.plot_quick_ts(ts_counts_tech, 'counts')

alt.Chart(...)

In [90]:
au.ts_magnitude_growth_(ts_counts_tech, year_start = 2019, year_end = 2023)

,magnitude,growth
counts,389.8,0.974314


### Distribution of different technologies

In [91]:
tech_subtype_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    .query("type == 'Technology'")
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'subtype'])
)

In [92]:
# Total tech funding
counts_total = tech_subtype_df.drop_duplicates('id').query("year >= 2019").id.nunique()

In [93]:
tech_subtype_dist = (
    tech_subtype_df
    .query("year >= 2019")
    .groupby('subtype')
    .agg(
        counts=('id', 'nunique'), 
    )
    .reset_index()
    .assign(counts_prop = lambda df: df.counts/counts_total)
)

tech_subtype_dist

,subtype,counts,counts_prop
0,AI,1257,0.644946
1,Immersive tech,750,0.384813
2,Internet,9,0.004618
3,Mobile,387,0.198563


### Growth of technology topics

In [94]:
column = 'subtype'
value = 'counts'

tech_subtype_ts = (
    tech_subtype_df
    .drop_duplicates(['id', column])
    .groupby(['subtype', 'year'])
    .agg(
        counts=('id', 'nunique'), 
    )
    .reset_index()
)

tech_subtype_ts = utils.impute_empty_periods_all_ts(tech_subtype_ts, column)

utils.magnitude_and_growth(tech_subtype_ts, column, value)

,magnitude,growth,subtype
0,251.4,5.485232,AI
0,150.0,-13.191489,Immersive tech
0,1.8,-40.000000,Internet
0,77.4,-35.220126,Mobile


In [95]:
fig = pu.ts_smooth(
    tech_subtype_ts,
    ["AI", "Immersive tech", "Internet", "Mobile"],
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 2: Applications

- Where are these technologies applied the most?
- Where do we see growth vs stagnation when it comes to applications?

In [154]:
tech_ids = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2013")
    .drop_duplicates('id')
    .id.to_list()
)

tech_ids_5y = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2019")
    .drop_duplicates('id')
    .id.to_list()
)

### Application distribution

In [155]:
column = 'type'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology'"),
    column=column, 
    values=['id'],
    ts=True
)


In [156]:
tech_applications_df

,type,counts,counts_prop
0,Biosciences,69,0.035
1,Child care & preschool,76,0.039
2,Development & learning,98,0.05
3,General,1565,0.803
4,Health,589,0.302
5,Parenting,5,0.003
6,Social,2,0.001
7,Technology,1949,1.0


In [99]:
fig = pu.ts_smooth(
    tech_applications_ts,
    tech_applications_ts[column].unique(),
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

In [100]:
utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id')

,magnitude,growth,type,counts
6,0.4,inf,Social,2
5,13.8,205.882353,Biosciences,69
1,19.6,72.222222,Development & learning,98
7,1.0,50.000000,Parenting,5
2,313.0,2.346369,General,1565
4,389.8,0.974314,Technology,1949
3,117.8,-4.722222,Health,589
0,15.2,-9.090909,Child care & preschool,76


In [101]:
pd.set_option('display.max_colwidth', 200)
(
    data_exploded_df
    .query('id in @tech_ids')
    # .query("subtype == 'Personal social emotional'")
    .query("type == 'Social'")
    .drop_duplicates(['id'])
    .sort_values('year', ascending=False)
)[['id', 'text', 'topics', 'year']]

,id,text,topics,year
6108,KR-102267147-B1,"A system which manages the use time of child-care institution. The present invention relates to a childcare institution use time management system, and in the present invention, the &lt;informatio...",social_services,2021
20885,KR-20210111483-A,Happy communication support system in connection with kindergarten and nursery school. The present invention is a communication web/app for parents and teachers of kindergartens and daycare center...,social_services,2021
5567,CN-102930409-B,Application service system based on infant formula milk intake and development status. The invention discloses an application service system based on infant formula milk intake and development sta...,social_services,2016


### Application distribution: More granular subtypes

In [116]:
column = 'subtype'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology'"),
    column=column, 
    values=['id'],
    ts=True
)
tech_applications_df.query("type != 'Technology'").sort_values('counts', ascending=False)

,subtype,counts,counts_prop,type
8,Infancy,1501,0.77,General
23,Sleep,335,0.172,Health
20,Physical development,147,0.075,Health
6,Health,102,0.052,Health
4,Games,80,0.041,General
14,Neuroscience,68,0.035,Biosciences
22,Preschool,66,0.034,Child care & preschool
21,Prenatal,38,0.019,Health
25,Special educational needs,34,0.017,Development & learning
15,Non-tech assessments,32,0.016,General


In [118]:
(
    utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id')
    # .sort_values(['type', 'growth'], ascending=False)
    .sort_values('growth', ascending=False)
)

,magnitude,growth,subtype,counts,type
0,0.4,inf,Social services,2,Social
1,5.8,1200.000000,Communication and language,29,Development & learning
2,2.4,400.000000,Cognitive development,12,Development & learning
3,6.4,271.428571,Non-tech assessments,32,General
4,13.6,200.000000,Neuroscience,68,Biosciences
5,2.4,125.000000,Nutrition & weight,12,Health
6,7.6,78.571429,Prenatal,38,Health
7,6.8,66.666667,Special educational needs,34,Development & learning
8,3.4,66.666667,Literacy,17,Development & learning
9,1.0,50.000000,Parenting,5,Parenting


In [105]:
cat_type = 'Development & learning'
cats = list(topics_df.query("type == @cat_type").subtype.unique())

In [106]:
fig = pu.ts_smooth(
    tech_applications_ts,
    cats,
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

### Which technology is applied to most to subtype X?

## Insight 3: Geographical insights

- Top countries in terms of counts
- UK vs baseline growth for overall counts, in technology counts and application counts

In [205]:
# data_exploded_df.explode('country_code').drop_duplicates(['id', 'country_code']).isnull().sum()
# Consider lack of information

In [108]:
data_countries_df = (
    data_exploded_df
    # .explode('country_code')
    .dropna(subset=['country_code'])
    .query("type == 'Technology'")
    .query('subtype in @tech_subtypes')
    .drop_duplicates(['id'])    
)
country_codes = data_countries_df.country_code.unique()

growth_df = []
ts_counts = []
for country_code in country_codes:
    country_df = data_countries_df.query("country_code == @country_code")
    _ts_counts = utils.get_timeseries(country_df, column='id').assign(country_code = country_code)
    growth_df.append(
        au.ts_magnitude_growth_(
            ts_df = _ts_counts,
            year_start = 2019,
            year_end = 2023  
        )
        .assign(country_code = country_code)
        .reset_index(drop=True)
    )
    ts_counts.append(_ts_counts)
growth_df = pd.concat(growth_df, ignore_index=True)
ts_counts = pd.concat(ts_counts, ignore_index=True)

In [111]:
(
    growth_df
    .sort_values('magnitude', ascending=False)
    .head(20)
)

,magnitude,growth,country_code
0,250.6,-8.729140,CN
5,53.0,44.166667,KR
3,30.4,-13.978495,US
4,16.4,-14.285714,WO
29,9.4,29.166667,JP
1,5.8,120.000000,EP
9,4.2,40.000000,TW
23,3.4,700.000000,AU
13,2.6,100.000000,TR
14,2.2,700.000000,DE


In [110]:
countries = ['US', 'GB', 'CN', 'KR', 'JP']
fig = pu.ts_smooth(
    ts_counts.query("country_code in @countries"),
    countries,
    variable= "counts",
    variable_title = "",
    category_column = 'country_code',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

In [61]:
data_countries_df = (
    data_exploded_df
    .explode('country_code')
    .dropna(subset=['country_code'])
    .query("type == 'Technology'")
    .query('subtype in @tech_subtypes')
    .drop_duplicates(['id', 'subtype']) 
    .query("country_code == 'GB'")   
)

In [62]:
data_countries_df.groupby('subtype').agg(counts=('id', 'nunique')).reset_index()

,subtype,counts
0,AI,9
1,Immersive tech,2
2,Mobile,4
